# 01 — Battlefield Data Exploration
Real data from OSM, USGS, ADS-B, OpenWeatherMap, Sentinel-2, CoT

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
from data.osm_terrain_fetcher import OSMTerrainFetcher
import folium

fetcher = OSMTerrainFetcher()
bbox = (34.0, -117.5, 34.5, -117.0)
roads = fetcher.fetch_roads(bbox)
print(f'Roads fetched: {len(roads)} features')

m = folium.Map(location=[34.25, -117.25], zoom_start=10, tiles='CartoDB dark_matter')
for _, road in roads.iterrows():
    if hasattr(road.geometry, 'coords'):
        coords = [(c[1], c[0]) for c in road.geometry.coords]
        folium.PolyLine(coords, color='#00ff41', weight=1).add_to(m)
m

In [ ]:
from data.usgs_dem_fetcher import USGSDEMFetcher
import plotly.graph_objects as go
import numpy as np

dem_fetcher = USGSDEMFetcher()
dem = dem_fetcher.fetch_dem(bbox)
print(f'DEM shape: {dem.shape}, range: [{dem.min():.0f}, {dem.max():.0f}]m')

fig = go.Figure(go.Surface(z=dem, colorscale='earth'))
fig.update_layout(title='USGS DEM — 3D Elevation', width=800, height=500,
                  scene=dict(zaxis_title='Elevation (m)'))
fig.show()

In [ ]:
from terrain.slope_calculator import SlopeCalculator
from terrain.los_calculator import LOSCalculator
import plotly.express as px

calc = SlopeCalculator()
slope = calc.compute_slope(dem)
traff = calc.compute_trafficability(slope)
stats = calc.get_statistics(slope)
print(f'Slope stats: {stats}')

fig = px.imshow(traff, color_continuous_scale=['#00ff41','#ffcc00','#ff3333'],
                title='Trafficability: GO / SLOW-GO / NO-GO')
fig.show()

In [ ]:
from data.adsb_fetcher import ADSBFetcher

adsb = ADSBFetcher()
aircraft = adsb.fetch_live_aircraft(bbox)
print(f'Aircraft in AO: {len(aircraft)}')
for ac in aircraft[:5]:
    print(f'  {ac.callsign}: {ac.latitude:.2f}, {ac.longitude:.2f}, alt={ac.altitude_m:.0f}m')

In [ ]:
from data.weather_fetcher import WeatherFetcher

weather = WeatherFetcher()
current = weather.fetch_current(34.25, -117.25)
print(f'Weather: {current}')
forecast = weather.fetch_forecast(34.25, -117.25)
print(f'Forecast entries: {len(forecast)}')

In [ ]:
from data.cot_parser import CoTParser

parser = CoTParser()
sample_xml = CoTParser.generate_sample_cot('B01', 'WARHORSE-1', 'a-f-G', 34.05, -117.45)
event = parser.parse_event(sample_xml)
print(f'CoT Event: uid={event.uid}, affiliation={event.affiliation}')
print(f'Position: {event.lat}, {event.lon}')

In [ ]:
from sensors.sensor_aggregator import SensorAggregator

agg = SensorAggregator()
for _ in range(50):
    agg.update_imu(0, 0, 1, 0, 0, 0)
agg.update_gps(34.05, -117.45, 900)
state = agg.get_fused_state()
print(f'Fused position: ({state.position.latitude:.4f}, {state.position.longitude:.4f})')
print(f'Motion: {state.motion_state}, Agreement: {state.agreement_score:.2f}')

In [ ]:
from evaluation.metrics import C2Metrics
import numpy as np

m = C2Metrics()
rng = np.random.default_rng(42)
for _ in range(200):
    m.record_latency(rng.exponential(40))
    m.record_message('battlefield.state')
stats = m.c2_latency_stats()
print(f'NATS throughput: {m.throughput_stats()}')
print(f'Latency: mean={stats["mean"]:.1f}ms p95={stats["p95"]:.1f}ms')